<a href="https://colab.research.google.com/github/kukanmani06/Internship_Projects/blob/main/NLP_BOW_%26_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk

In [ ]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.stem import wordnet
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import pos_tag
from sklearn.metrics.pairwise import pairwise_distances
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [ ]:
df = pd.read_excel('/content/dialog_agent.xlsx')

In [ ]:
df.head()

,Context,Text Response
0,Tell me about your personality,Just think of me as the ace up your sleeve.
1,I want to know you better,I can help you work smarter instead of harder
2,Define yourself,NaN
3,Describe yourself,NaN
4,tell me about yourself,NaN


In [ ]:
df.shape

(1592, 2)

In [ ]:
df.ffill(axis=0,inplace=True)

In [ ]:
df.head(10)

,Context,Text Response
0,Tell me about your personality,Just think of me as the ace up your sleeve.
1,I want to know you better,I can help you work smarter instead of harder
2,Define yourself,I can help you work smarter instead of harder
3,Describe yourself,I can help you work smarter instead of harder
4,tell me about yourself,I can help you work smarter instead of harder
5,all about you,I can help you work smarter instead of harder
6,tell me some stuff about you,I can help you work smarter instead of harder
7,talk some stuff about you,I can help you work smarter instead of harder
8,talk about yourself,I can help you work smarter instead of harder
9,about yourself,I can help you work smarter instead of harder


In [ ]:
def text_normalization(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z]', ' ', text)
    tokens = word_tokenize(text)
    lemmas = wordnet.WordNetLemmatizer()
    tags_list = pos_tag(tokens, tagset = None)
    lemma_words = []
    for token, pos_token in tags_list:
        if pos_token.startswith('V'):       # Verb
            pos_val = 'v'
        elif pos_token.startswith('J'):     # Adjective
            pos_val = 'a'
        elif pos_token.startswith('R'):     # Adverb
            pos_val = 'r'
        else:
            pos_val = 'n'
        lemma_token = lemmas.lemmatize(token, pos_val)
        lemma_words.append(lemma_token)
    clean_text = " ".join(lemma_words)
    return clean_text

In [ ]:
df['Lemmatized_Context'] = df['Context'].apply(text_normalization)

In [ ]:
df.head()

,Context,Text Response,Lemmatized_Context
0,Tell me about your personality,Just think of me as the ace up your sleeve.,tell me about your personality
1,I want to know you better,I can help you work smarter instead of harder,i want to know you good
2,Define yourself,I can help you work smarter instead of harder,define yourself
3,Describe yourself,I can help you work smarter instead of harder,describe yourself
4,tell me about yourself,I can help you work smarter instead of harder,tell me about yourself


Bag of Words
*  It is a representation technique where - Text is converted into numbers.

In [ ]:
cv = CountVectorizer()
x = cv.fit_transform(df['Lemmatized_Context']).toarray()

In [ ]:
x

array([[0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 1, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
feature_names = cv.get_feature_names_out()

In [ ]:
x_df = pd.DataFrame(x, columns=feature_names)

In [ ]:
x_df

,abort,about,absolutely,abysmal,actually,adore,advice,advise,affirmative,afraid,...,year,yeh,yep,yes,yet,you,your,yours,yourself,yup
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1588,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1589,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1590,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
Question = "Will you help me and tell me about yourself."

In [ ]:
stop_words = set(stopwords.words('english'))
Ques=[]
Ques_Split = Question.lower().split()
for word in Ques_Split:
  if word  in stop_words:
    continue
  else:
    Ques.append(word)
clean_ques = " ".join(Ques)
print(clean_ques)

help tell yourself.


In [ ]:
Question_lemma = text_normalization(clean_ques)
Question_bow = cv.transform([Question_lemma]).toarray()

In [ ]:
Question_bow

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
cosine_value = 1 - pairwise_distances(x_df, Question_bow, metric = 'cosine')
cosine_value

array([[0.25819889],
       [0.        ],
       [0.40824829],
       ...,
       [0.        ],
       [0.        ],
       [0.        ]])

In [ ]:
df['Similarity'] = cosine_value

In [ ]:
df_similarity = pd.DataFrame(df, columns=['Text Response', 'Similarity'])
df_similarity

,Text Response,Similarity
0,Just think of me as the ace up your sleeve.,0.258199
1,I can help you work smarter instead of harder,0.000000
2,I can help you work smarter instead of harder,0.408248
3,I can help you work smarter instead of harder,0.408248
4,I can help you work smarter instead of harder,0.577350
...,...,...
1587,Talking is what I do best.,0.000000
1588,I'll be waiting.,0.000000
1589,All right. I'll be here.,0.000000
1590,Till next time.,0.000000


In [ ]:
df_similarity_sort = df_similarity.sort_values(by='Similarity', ascending=False)
df_similarity_sort.head()

,Text Response,Similarity
4,I can help you work smarter instead of harder,0.577350
211,I'm glad to help. What can I do for you?,0.577350
186,I'm glad to help. What can I do for you?,0.408248
2,I can help you work smarter instead of harder,0.408248
9,I can help you work smarter instead of harder,0.408248


In [ ]:
df.sort_values(by='Similarity',ascending=False).head(5)

,Context,Text Response,Lemmatized_Context,Similarity
4,tell me about yourself,I can help you work smarter instead of harder,tell me about yourself,0.577350
211,help,I'm glad to help. What can I do for you?,help,0.577350
186,I need help,I'm glad to help. What can I do for you?,i need help,0.408248
2,Define yourself,I can help you work smarter instead of harder,define yourself,0.408248
9,about yourself,I can help you work smarter instead of harder,about yourself,0.408248


In [ ]:
threshold = 0.2
df_threshold = df_similarity_sort[df_similarity_sort['Similarity'] > threshold]
df_threshold.head()

,Text Response,Similarity
4,I can help you work smarter instead of harder,0.577350
211,I'm glad to help. What can I do for you?,0.577350
186,I'm glad to help. What can I do for you?,0.408248
2,I can help you work smarter instead of harder,0.408248
9,I can help you work smarter instead of harder,0.408248


TF-IDF Model

* Term Frequency & Inverse-Document Frequency is a text vectorization technique used in NLP to measure how imporatant a word is to a document within a collection of documents (corpus).
Term-Frequenct (TF): How often a word appears in a document

* TF(t,d) = Number of times term (t) appears in document (d​) / Total words in document (d)
Inverse-Docuemnt Frequency (IDF): How rare a word is across all documents.

* IDF(t) = log(Total documents / Documents containing term t​)

In [ ]:
Question = "Will you help me and tell me about yourself."

In [ ]:
tfidf = TfidfVectorizer()
x_tfidf = tfidf.fit_transform(df['Lemmatized_Context']).toarray()

In [ ]:
question_lemma = text_normalization(Question)
question_tfidf = tfidf.transform([question_lemma]).toarray()

In [ ]:
df_tfidf = pd.DataFrame(x_tfidf, columns=tfidf.get_feature_names_out())
df_tfidf

,abort,about,absolutely,abysmal,actually,adore,advice,advise,affirmative,afraid,...,year,yeh,yep,yes,yet,you,your,yours,yourself,yup
0,0.0,0.407572,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.330555,0.0,0.000000,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.204515,0.000000,0.0,0.000000,0.0
2,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.641790,0.0
3,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.641790,0.0
4,0.0,0.453790,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.608937,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0
1588,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0
1589,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0
1590,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.122823,0.000000,0.0,0.000000,0.0


In [ ]:
df.sort_values(by='Similarity',ascending=False).head(5)

,Context,Text Response,Lemmatized_Context,Similarity
4,tell me about yourself,I can help you work smarter instead of harder,tell me about yourself,0.577350
211,help,I'm glad to help. What can I do for you?,help,0.577350
186,I need help,I'm glad to help. What can I do for you?,i need help,0.408248
2,Define yourself,I can help you work smarter instead of harder,define yourself,0.408248
9,about yourself,I can help you work smarter instead of harder,about yourself,0.408248


In [ ]:
cosine_value = 1 - pairwise_distances(df_tfidf, question_tfidf, metric = 'cosine')
cosine_value

array([[0.44023004],
       [0.02321557],
       [0.25293815],
       ...,
       [0.        ],
       [0.01394226],
       [0.        ]])

In [ ]:
df['Similarity_tfidf'] = cosine_value

In [ ]:
df_simi_tfidf = pd.DataFrame(df, columns=['Text Response','Similarity_tfidf'])
df_simi_tfidf

,Text Response,Similarity_tfidf
0,Just think of me as the ace up your sleeve.,0.440230
1,I can help you work smarter instead of harder,0.023216
2,I can help you work smarter instead of harder,0.252938
3,I can help you work smarter instead of harder,0.252938
4,I can help you work smarter instead of harder,0.730142
...,...,...
1587,Talking is what I do best.,0.000000
1588,I'll be waiting.,0.000000
1589,All right. I'll be here.,0.000000
1590,Till next time.,0.013942


In [ ]:
df_simi_tfidf_sort = df_simi_tfidf.sort_values(by='Similarity_tfidf', ascending=False)
df_simi_tfidf_sort.head(10)

,Text Response,Similarity_tfidf
4,I can help you work smarter instead of harder,0.730142
194,I'm glad to help. What can I do for you?,0.647028
16,I can help you work smarter instead of harder,0.627863
214,I'm glad to help. What can I do for you?,0.528215
184,I'm glad to help. What can I do for you?,0.517553
413,Absolutely. You don't have to ask.,0.500388
9,I can help you work smarter instead of harder,0.491513
379,I should get one. It's all work and no play la...,0.463108
500,The virtual world is my playground. I'm always...,0.459988
538,My pleasure.,0.458173


In [ ]:
threshold = 0.1 # considering the value of p=smiliarity to be greater than 0.2
df_threshold = df_simi_tfidf_sort[df_simi_tfidf_sort['Similarity_tfidf'] > threshold]
df_threshold

,Text Response,Similarity_tfidf
4,I can help you work smarter instead of harder,0.730142
194,I'm glad to help. What can I do for you?,0.647028
16,I can help you work smarter instead of harder,0.627863
214,I'm glad to help. What can I do for you?,0.528215
184,I'm glad to help. What can I do for you?,0.517553
...,...,...
7,I can help you work smarter instead of harder,0.135762
1518,Probably I won't be able to give you the right...,0.131332
776,Cancelled! Let me know what I should do next.,0.126546
1016,"Ok, let's not talk about it then.",0.119119


In [ ]:
# defining a function that returns response to query using tf-idf
def chat_tfidf(text):
    lemma=text_normalization(text) # calling the function to perform text normalization
    tf=tfidf.transform([lemma]).toarray() # applying tf-idf
    cos=1-pairwise_distances(df_tfidf,tf,metric='cosine') # applying cosine similarity
    index_value=cos.argmax() # getting index value
    return df['Text Response'].loc[index_value]

In [ ]:
chat_tfidf('hi')

'Hey!'

In [ ]:
chat_tfidf('how are you?')

'Lovely, thanks.'

In [ ]:
chat_tfidf("Who are you ?")

'I can help you work smarter instead of harder'

In [ ]:
chat_tfidf("I'd like to know your age")

"I'm a relatively new bot, but I'm wise beyond my years."

In [ ]:
chat_tfidf("Can you write poem for me.")

"I'm glad to help. What can I do for you?"

In [ ]:
chat_tfidf("Write poem about birds.")

'Lovely, thanks.'

In [ ]:
chat_tfidf("Can you play a song for me.")

'Very funny, boss.'